# 🔍 BOSS Datalake — Source (Postgres) vs Bronze (Trino) Validator

**Purpose:** Source PostgreSQL table aur Bronze layer (Trino) ka data compare karna.

| Layer | Connection |
|-------|------------|
| **Source** | PostgreSQL (RDS) |
| **Bronze** | Trino (Datalake) |

### Checks Performed
| # | Check | Description |
|---|-------|-------------|
| 1 | **Column Count** | Source columns vs Bronze columns (pipeline cols exclude) |
| 2 | **Data Type Validation** | Source `information_schema` types vs Bronze column types |
| 3 | **Row Count** | Source rows (retention filter) vs Bronze rows |
| 4 | **Duplicate Check** | Bronze mein duplicates expected — verify |
| 5 | **NULL Validation** | Id join se — source NULL vs Bronze NULL (true/false null) |
| 6 | **Pipeline Columns Check** | Injected columns null nahi hone chahiye |

---
## 📦 Cell 1 — Libraries

In [ ]:
import pandas as pd
import urllib3
from IPython.display import display
from trino.dbapi import connect as trino_connect
from trino.auth import BasicAuthentication
import psycopg2
import psycopg2.extras

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 100)

print('✅ Libraries loaded!')

---
## ⚙️ Cell 2 — Configuration (Sirf yahan change karo)

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  SOURCE — PostgreSQL (RDS)
# ─────────────────────────────────────────────────────────────────
PG_CONFIG = {
    'host'    : 'your-rds-host.amazonaws.com',
    'port'    : 5432,
    'dbname'  : 'your_db',
    'user'    : 'your_user',
    'password': 'your_password'
}

# ─────────────────────────────────────────────────────────────────
#  TABLE CONFIG — boss_metadata se yeh values lo
# ─────────────────────────────────────────────────────────────────
SOURCE_TABLE     = 'TaxLot'           # boss_metadata.table_name
RETENTION_DAYS   = 90                 # boss_metadata.retention_days
PARTITION_COL    = 'SystemDate'       # boss_metadata.partition_col
PRIMARY_KEY      = 'Id'              # source table ka unique ID column

# ─────────────────────────────────────────────────────────────────
#  BRONZE — Trino
# ─────────────────────────────────────────────────────────────────
TRINO_CONFIG = {
    'host'    : '3.221.185.123',
    'port'    : 8443,
    'user'    : 'shariq.mehmood',
    'password': 'your-password',
    'catalog' : 'iceberg',
    'schema'  : 'boss'
}

BRONZE_TABLE = 'iceberg.boss.bronze_taxlot'   # bronze table full name

# ─────────────────────────────────────────────────────────────────
#  PIPELINE COLUMNS — ingestion ne inject kiye (schema mein nahi hote)
#  baad mein update karna agar aur columns hain
# ─────────────────────────────────────────────────────────────────
PIPELINE_COLUMNS = ['s3_path', 'ingestedfordate']

# NULL validation sample size
NULL_SAMPLE_SIZE = 1000

print(f'✅ Source Table   : {SOURCE_TABLE}')
print(f'✅ Bronze Table   : {BRONZE_TABLE}')
print(f'✅ Retention      : {RETENTION_DAYS} days on {PARTITION_COL}')
print(f'✅ Primary Key    : {PRIMARY_KEY}')
print(f'✅ Pipeline Cols  : {PIPELINE_COLUMNS}')

---
## 🔌 Cell 3 — Connections

In [ ]:
def get_pg_connection():
    return psycopg2.connect(**PG_CONFIG)


def get_trino_connection():
    auth = BasicAuthentication(TRINO_CONFIG['user'], TRINO_CONFIG['password'])
    return trino_connect(
        host        = TRINO_CONFIG['host'],
        port        = TRINO_CONFIG['port'],
        user        = TRINO_CONFIG['user'],
        auth        = auth,
        http_scheme = 'https',
        verify      = False,
        catalog     = TRINO_CONFIG['catalog'],
        schema      = TRINO_CONFIG['schema']
    )


# Test connections
try:
    pg_conn = get_pg_connection()
    print('✅ PostgreSQL connected!')
    pg_conn.close()
except Exception as e:
    print(f'❌ PostgreSQL connection failed: {e}')

try:
    tr_conn = get_trino_connection()
    cur = tr_conn.cursor()
    cur.execute('SELECT 1')
    cur.fetchall()
    print('✅ Trino connected!')
except Exception as e:
    print(f'❌ Trino connection failed: {e}')

---
## 📋 Cell 4 — Column Count Check (Source vs Bronze)

In [ ]:
def get_source_columns():
    """
    PostgreSQL information_schema se source table ke columns + types fetch karo.
    """
    pg_conn = get_pg_connection()
    query = """
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = %s
        ORDER BY ordinal_position
    """
    df = pd.read_sql(query, pg_conn, params=(SOURCE_TABLE,))
    pg_conn.close()
    df['column_name'] = df['column_name'].str.lower()
    return df


def get_bronze_columns():
    """
    Trino se Bronze table ke columns fetch karo.
    """
    tr_conn = get_trino_connection()
    cur     = tr_conn.cursor()
    cur.execute(f'DESCRIBE {BRONZE_TABLE}')
    rows = cur.fetchall()
    df   = pd.DataFrame(rows, columns=['column_name', 'data_type', 'extra', 'comment'])
    df['column_name'] = df['column_name'].str.lower()
    return df[['column_name', 'data_type']]


src_cols_df    = get_source_columns()
bronze_cols_df = get_bronze_columns()

src_cols_set    = set(src_cols_df['column_name'].tolist())
pipeline_norm   = set(c.lower() for c in PIPELINE_COLUMNS)
bronze_cols_set = set(bronze_cols_df['column_name'].tolist())

# Pipeline columns bronze se hatao comparison k liye
bronze_excl_pipeline = bronze_cols_set - pipeline_norm

matched          = src_cols_set & bronze_excl_pipeline
missing_in_bronze = src_cols_set - bronze_excl_pipeline
extra_in_bronze  = bronze_excl_pipeline - src_cols_set

print('=' * 65)
print('📋 COLUMN COUNT CHECK — SOURCE vs BRONZE')
print('=' * 65)
print(f'  Source columns              : {len(src_cols_set)}')
print(f'  Bronze columns (total)      : {len(bronze_cols_set)}')
print(f'  Pipeline columns (excluded) : {len(pipeline_norm)}')
print(f'  Bronze columns (excl pipe)  : {len(bronze_excl_pipeline)}')
print(f'  ✅ Matched                  : {len(matched)}')

print(f'\n🔴 Missing in Bronze ({len(missing_in_bronze)}) — source mein hai, bronze mein nahi:')
if missing_in_bronze:
    for i, col in enumerate(sorted(missing_in_bronze), 1):
        print(f'  {i}. {col}')
else:
    print('  ✅ None')

print(f'\n🔵 Extra in Bronze ({len(extra_in_bronze)}) — bronze mein hai, source mein nahi:')
if extra_in_bronze:
    for i, col in enumerate(sorted(extra_in_bronze), 1):
        print(f'  {i}. {col}')
else:
    print('  ✅ None')

---
## 🔢 Cell 5 — Row Count Check (Source vs Bronze)

In [ ]:
def get_source_row_count():
    """
    Source se row count — retention filter ke saath.
    current_date - retention_days se pehle ka data.
    """
    pg_conn = get_pg_connection()
    query   = f"""
        SELECT COUNT(*) as cnt
        FROM "{SOURCE_TABLE}"
        WHERE "{PARTITION_COL}" < NOW() - INTERVAL '{RETENTION_DAYS} days'
    """
    cur = pg_conn.cursor()
    cur.execute(query)
    count = cur.fetchone()[0]
    pg_conn.close()
    return count


def get_bronze_row_count():
    """
    Bronze se total row count.
    """
    tr_conn = get_trino_connection()
    cur     = tr_conn.cursor()
    cur.execute(f'SELECT COUNT(*) FROM {BRONZE_TABLE}')
    return cur.fetchone()[0]


def get_bronze_duplicate_count():
    """
    Bronze mein duplicate Id count — duplicates expected hain.
    """
    tr_conn = get_trino_connection()
    cur     = tr_conn.cursor()
    query   = f"""
        SELECT COUNT(*) - COUNT(DISTINCT "{PRIMARY_KEY}") as duplicate_count
        FROM {BRONZE_TABLE}
    """
    cur.execute(query)
    return cur.fetchone()[0]


src_row_count    = get_source_row_count()
bronze_row_count = get_bronze_row_count()
bronze_dup_count = get_bronze_duplicate_count()
bronze_unique    = bronze_row_count - bronze_dup_count

match = src_row_count == bronze_unique

print('=' * 65)
print('🔢 ROW COUNT CHECK — SOURCE vs BRONZE')
print('=' * 65)
print(f'  Source rows (retention filter) : {src_row_count}')
print(f'  Bronze rows (total)            : {bronze_row_count}')
print(f'  Bronze duplicates              : {bronze_dup_count}')
print(f'  Bronze unique rows             : {bronze_unique}')
print()
print(f'  Source == Bronze unique        : {"✅ MATCH" if match else "❌ MISMATCH"}')

if not match:
    diff = bronze_unique - src_row_count
    print(f'  Difference                     : {diff:+d}')

---
## 🧪 Cell 6 — Data Type Validation (Source vs Bronze)

In [ ]:
# Postgres type → normalized type map
PG_TYPE_MAP = {
    'integer'                   : 'integer',
    'bigint'                    : 'bigint',
    'smallint'                  : 'smallint',
    'numeric'                   : 'decimal',
    'decimal'                   : 'decimal',
    'double precision'          : 'double',
    'real'                      : 'double',
    'character varying'         : 'varchar',
    'character'                 : 'varchar',
    'text'                      : 'varchar',
    'boolean'                   : 'boolean',
    'date'                      : 'date',
    'timestamp without time zone': 'timestamp',
    'timestamp with time zone'  : 'timestamp',
    'uuid'                      : 'varchar',
    'json'                      : 'varchar',
    'jsonb'                     : 'varchar',
}

# Trino type → normalized type map
TRINO_TYPE_MAP = {
    'integer'  : 'integer',
    'bigint'   : 'bigint',
    'smallint' : 'smallint',
    'decimal'  : 'decimal',
    'double'   : 'double',
    'varchar'  : 'varchar',
    'boolean'  : 'boolean',
    'date'     : 'date',
    'timestamp': 'timestamp',
}


def normalize_trino_type(raw):
    if not raw:
        return 'unknown'
    base = raw.lower().split('(')[0].strip()
    return TRINO_TYPE_MAP.get(base, base)


def normalize_pg_type(raw):
    if not raw:
        return 'unknown'
    return PG_TYPE_MAP.get(raw.lower().strip(), raw.lower().strip())


print('=' * 80)
print('🧪 DATA TYPE VALIDATION — SOURCE vs BRONZE')
print('=' * 80)
print(f'  {"COLUMN":<35} | {"SOURCE TYPE":<20} | {"BRONZE TYPE":<20} | STATUS')
print(f'  {"-" * 80}')

src_type_map    = dict(zip(src_cols_df['column_name'], src_cols_df['data_type']))
bronze_type_map = dict(zip(bronze_cols_df['column_name'], bronze_cols_df['data_type']))

dtype_results = []
for col in sorted(matched):
    src_raw    = src_type_map.get(col, 'unknown')
    bronze_raw = bronze_type_map.get(col, 'unknown')
    src_norm   = normalize_pg_type(src_raw)
    bronze_norm = normalize_trino_type(bronze_raw)
    match      = src_norm == bronze_norm
    status     = '✅ MATCH' if match else '❌ MISMATCH'
    print(f'  {col:<35} | {src_raw:<20} | {bronze_raw:<20} | {status}')
    dtype_results.append({'column': col, 'source_type': src_raw,
                           'bronze_type': bronze_raw, 'status': status})

dtype_df = pd.DataFrame(dtype_results)
mismatch_count = len(dtype_df[dtype_df['status'] == '❌ MISMATCH'])
print(f'  {"-" * 80}')
print(f'\n  ✅ Match    : {len(dtype_df) - mismatch_count}/{len(dtype_df)}')
print(f'  ❌ Mismatch : {mismatch_count}/{len(dtype_df)}')

---
## 🔬 Cell 7 — NULL Validation (Id join — True NULL vs False NULL)

In [ ]:
def fetch_source_sample(sample_size):
    """
    Source se sample fetch karo — retention filter ke saath.
    """
    pg_conn = get_pg_connection()
    query   = f"""
        SELECT *
        FROM "{SOURCE_TABLE}"
        WHERE "{PARTITION_COL}" < NOW() - INTERVAL '{RETENTION_DAYS} days'
        LIMIT {sample_size}
    """
    df = pd.read_sql(query, pg_conn)
    pg_conn.close()
    df.columns = df.columns.str.lower()
    print(f'✅ Source: {len(df)} rows fetched')
    return df


def fetch_bronze_sample(id_list):
    """
    Bronze se sirf wahi rows fetch karo jo source sample mein hain — Id se.
    """
    tr_conn = get_trino_connection()
    cur     = tr_conn.cursor()
    ids_str = ', '.join(str(i) for i in id_list)
    query   = f"""
        SELECT *
        FROM {BRONZE_TABLE}
        WHERE "{PRIMARY_KEY}" IN ({ids_str})
    """
    cur.execute(query)
    rows = cur.fetchall()
    cols = [desc[0].lower() for desc in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    # Duplicates hain bronze mein — latest ingestedfordate rakho per Id
    if 'ingestedfordate' in df.columns:
        df = df.sort_values('ingestedfordate', ascending=False)
        df = df.drop_duplicates(subset=[PRIMARY_KEY.lower()], keep='first')
    print(f'✅ Bronze: {len(df)} rows fetched (deduped by Id)')
    return df


def validate_nulls(src_df, bronze_df):
    """
    Id join ke baad column-wise NULL check.
    TRUE NULL  = source mein bhi NULL tha
    FALSE NULL = source mein value thi, bronze mein NULL
    """
    pk = PRIMARY_KEY.lower()
    src_df    = src_df.copy()
    bronze_df = bronze_df.copy()
    src_df[pk]    = src_df[pk].astype(str).str.strip()
    bronze_df[pk] = bronze_df[pk].astype(str).str.strip()

    merged = src_df.merge(bronze_df, on=pk, suffixes=('_src', '_bronze'))
    print(f'\n   Source rows  : {len(src_df)}')
    print(f'   Bronze rows  : {len(bronze_df)}')
    print(f'   Joined rows  : {len(merged)}')

    unmatched = len(src_df) - len(merged)
    if unmatched > 0:
        print(f'   ⚠️  Unmatched : {unmatched} rows (Id Trino mein nahi mili)')

    skip_cols = set(PIPELINE_COLUMNS) | {pk, 'ingestedfordate', 's3_path'}
    src_cols_in_merge = [c for c in merged.columns
                         if c.endswith('_src') and c.replace('_src', '') not in skip_cols]
    check_cols = [c.replace('_src', '') for c in src_cols_in_merge]

    print(f'\n  {"COLUMN":<35} | {"BRONZE NULL":>11} | {"SRC NULL":>8} | {"FALSE NULL":>10} | STATUS')
    print(f'  {"-" * 90}')

    results          = []
    false_null_detail = []

    for col in sorted(check_cols):
        src_col    = col + '_src'
        bronze_col = col + '_bronze'
        if src_col not in merged.columns or bronze_col not in merged.columns:
            continue

        t_vals = merged[bronze_col]
        s_vals = merged[src_col]

        bronze_null_mask  = t_vals.isna() | t_vals.astype(str).str.strip().isin(
                            ['', 'None', 'null', 'nan', 'NaN'])
        src_null_mask     = s_vals.isna() | s_vals.astype(str).str.strip().isin(
                            ['', 'None', 'null', 'nan', 'NaN'])
        false_null_mask   = bronze_null_mask & ~src_null_mask

        bronze_null_count = int(bronze_null_mask.sum())
        src_null_count    = int(src_null_mask.sum())
        false_null_count  = int(false_null_mask.sum())

        if bronze_null_count == 0:
            status = '⏭️  SKIP'
        elif false_null_count > 0:
            status = '❌ FALSE NULL'
            false_rows = merged[false_null_mask][[pk, src_col]].copy()
            if false_null_count < 10:
                for _, r in false_rows.iterrows():
                    false_null_detail.append({
                        'column'      : col,
                        PRIMARY_KEY   : r[pk],
                        'source_value': str(r[src_col])[:50]
                    })
            else:
                false_null_detail.append({
                    'column'      : col,
                    PRIMARY_KEY   : f'{false_null_count} rows',
                    'source_value': 'multiple'
                })
        else:
            status = '✅ TRUE NULL'

        if bronze_null_count > 0:
            print(f'  {col:<35} | {bronze_null_count:>11} | {src_null_count:>8} | '
                  f'{false_null_count:>10} | {status}')

        results.append({'column': col, 'bronze_null': bronze_null_count,
                         'src_null': src_null_count, 'false_null': false_null_count,
                         'status': status})

    print(f'  {"-" * 90}')

    if false_null_detail:
        print(f'\n❌ FALSE NULL DETAIL:')
        print(f'  {"COLUMN":<35} | {PRIMARY_KEY:<20} | SOURCE VALUE')
        print(f'  {"-" * 80}')
        for r in false_null_detail:
            print(f'  {str(r["column"]):<35} | {str(r[PRIMARY_KEY]):<20} | {str(r["source_value"])[:40]}')
    else:
        print('\n✅ Koi FALSE NULL nahi!')

    return pd.DataFrame(results), pd.DataFrame(false_null_detail) if false_null_detail else pd.DataFrame()


print('=' * 90)
print('🔬 NULL VALIDATION — Id JOIN (True NULL vs False NULL)')
print('=' * 90)

src_sample_df    = fetch_source_sample(NULL_SAMPLE_SIZE)
id_list          = src_sample_df[PRIMARY_KEY.lower()].tolist()
bronze_sample_df = fetch_bronze_sample(id_list)
null_results_df, false_null_df = validate_nulls(src_sample_df, bronze_sample_df)

---
## 📊 Cell 9 — Final Summary

In [ ]:
col_match     = len(missing_in_bronze) == 0 and len(extra_in_bronze) == 0
row_match     = src_row_count == bronze_unique
dtype_match   = mismatch_count == 0
false_nulls   = len(false_null_df) > 0 if len(false_null_df) else False
pipeline_pass = all(r == '✅ PASS' for r in pipeline_df['status'].tolist()) if len(pipeline_df) else True

checks = [
    ('Column Count',        '✅ PASS' if col_match   else '❌ FAIL',
     f'Missing: {len(missing_in_bronze)}, Extra: {len(extra_in_bronze)}'),
    ('Row Count',           '✅ PASS' if row_match   else '❌ FAIL',
     f'Source: {src_row_count}, Bronze unique: {bronze_unique}, Duplicates: {bronze_dup_count}'),
    ('Data Types',          '✅ PASS' if dtype_match else '❌ FAIL',
     f'Mismatches: {mismatch_count}'),
    ('NULL Validation',     '✅ PASS' if not false_nulls else '❌ FAIL',
     f'False nulls: {len(false_null_df)} columns'),
    ('Pipeline Columns',    '✅ PASS' if pipeline_pass else '❌ FAIL',
     f'{len(pipeline_df)} columns checked'),
]

print('=' * 75)
print('📊 FINAL SUMMARY — SOURCE vs BRONZE')
print('=' * 75)
print(f'  {"CHECK":<25} | {"STATUS":<12} | DETAIL')
print(f'  {"-" * 70}')
for check, status, detail in checks:
    print(f'  {check:<25} | {status:<12} | {detail}')
print(f'  {"-" * 70}')
print(f'  Source Table  : {SOURCE_TABLE}')
print(f'  Bronze Table  : {BRONZE_TABLE}')
print('=' * 75)

if all(s == '✅ PASS' for _, s, _ in checks):
    print('\n🎉 ALL CHECKS PASSED!')
else:
    print('\n❌ SOME CHECKS FAILED — Details upar dekho!')